# LESSON 4.2: 2-D DFT and Spectrum Visualization
## Filtering in the Frequency Domain

In this lesson:
- The 2-D Discrete Fourier Transform
- Fourier spectrum and phase angle of images
- Centering the DFT using $(-1)^{x+y}$
- Properties of the 2-D DFT (translation, rotation, periodicity)
- Importance of phase vs magnitude

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The 2-D Discrete Fourier Transform

For a digital image $f(x, y)$ of size $M \times N$:

### Forward DFT:
$$F(u, v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x, y) \, e^{-j2\pi(ux/M + vy/N)}$$

### Inverse DFT:
$$f(x, y) = \frac{1}{MN} \sum_{u=0}^{M-1} \sum_{v=0}^{N-1} F(u, v) \, e^{j2\pi(ux/M + vy/N)}$$

### Key quantities:
- **Spectrum**: $|F(u,v)| = \sqrt{R^2(u,v) + I^2(u,v)}$
- **Phase angle**: $\phi(u,v) = \arctan\left(\frac{I(u,v)}{R(u,v)}\right)$
- **Power spectrum**: $P(u,v) = |F(u,v)|^2$

In [ ]:
# Create a simple test image: a rectangle
M, N = 256, 256
img = np.zeros((M, N), dtype=np.float64)
img[90:170, 110:150] = 255  # white rectangle

# Compute 2-D DFT
F = np.fft.fft2(img)

# Spectrum (magnitude) - not centered
spectrum = np.abs(F)

# Phase
phase = np.angle(F)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original Image f(x,y)', fontsize=12)

axes[1].imshow(np.log1p(spectrum), cmap='gray')
axes[1].set_title('Spectrum |F(u,v)| (NOT centered)\n(log scale)', fontsize=12)

axes[2].imshow(phase, cmap='gray')
axes[2].set_title('Phase Angle', fontsize=12)

plt.suptitle('2-D DFT of a Rectangle', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Centering the DFT

The raw DFT output has the DC component (zero frequency) at the **corners** of the array.

For visualization and filtering, we want the DC component at the **center**.

### Method: Multiply by $(-1)^{x+y}$ before computing DFT

$$f(x,y) \cdot (-1)^{x+y} \Leftrightarrow F(u - M/2, v - N/2)$$

This shifts the origin of $F(u,v)$ to the center of the frequency rectangle.

In NumPy, we use `np.fft.fftshift()` to achieve the same result after computing the DFT.

In [ ]:
# Centering the DFT
F = np.fft.fft2(img)
F_centered = np.fft.fftshift(F)  # shift zero-frequency to center

spectrum_uncentered = np.log1p(np.abs(F))
spectrum_centered = np.log1p(np.abs(F_centered))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original Image', fontsize=12)

axes[1].imshow(spectrum_uncentered, cmap='gray')
axes[1].set_title('Spectrum (NOT centered)\nDC at corners', fontsize=12)

axes[2].imshow(spectrum_centered, cmap='gray')
axes[2].set_title('Spectrum (CENTERED)\nDC at center', fontsize=12)

plt.suptitle('Centering the Fourier Transform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("DC component F(0,0) =", np.abs(F[0,0]).astype(int))
print("This equals the sum of all pixel values:", int(img.sum()))

## 3. The DC Component

The value at $F(0,0)$ is special:

$$F(0,0) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x,y) = MN \cdot \bar{f}$$

Where $\bar{f}$ is the **average intensity** of the image.

- $F(0,0)$ is called the **DC component** (from electrical engineering: "direct current")
- It is usually the **largest** component of the spectrum
- Setting it to zero removes the average intensity (image becomes darker)

In [ ]:
# Create a more complex test image
np.random.seed(42)
test_img = np.random.randint(50, 200, (256, 256), dtype=np.uint8).astype(np.float64)

# Add some structures
test_img[80:180, 80:180] = 220
test_img[120:140, 40:220] = 30

# Compute DFT
F = np.fft.fft2(test_img)

# Remove DC component
F_no_dc = F.copy()
F_no_dc[0, 0] = 0

# Reconstruct
img_no_dc = np.real(np.fft.ifft2(F_no_dc))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(test_img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'Original (avg = {test_img.mean():.1f})', fontsize=12)
axes[0].axis('off')

axes[1].imshow(np.log1p(np.abs(np.fft.fftshift(F))), cmap='gray')
axes[1].set_title('Centered Spectrum (log scale)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(img_no_dc, cmap='gray')
axes[2].set_title(f'DC removed (avg = {img_no_dc.mean():.1f})', fontsize=12)
axes[2].axis('off')

plt.suptitle('Effect of Removing the DC Component', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Log Transformation for Display

The DC component is usually **many orders of magnitude larger** than other frequency components.

To visualize the full spectrum, we use a **log transformation**:

$$D(u,v) = \log(1 + |F(u,v)|)$$

In [ ]:
# Create a rectangle and show spectrum with and without log
img_rect = np.zeros((256, 256), dtype=np.float64)
img_rect[78:178, 103:153] = 255

F = np.fft.fftshift(np.fft.fft2(img_rect))
spectrum = np.abs(F)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_rect, cmap='gray')
axes[0].set_title('Original Image', fontsize=12)

axes[1].imshow(spectrum, cmap='gray')
axes[1].set_title('Spectrum |F(u,v)| (linear scale)\nMost detail invisible', fontsize=12)

axes[2].imshow(np.log1p(spectrum), cmap='gray')
axes[2].set_title('log(1 + |F(u,v)|)\nDetails visible', fontsize=12)

plt.suptitle('Log Transformation for Spectrum Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Properties of the 2-D DFT

### Translation
$$f(x-x_0, y-y_0) \Leftrightarrow F(u,v) \, e^{-j2\pi(ux_0/M + vy_0/N)}$$

Translation does **not** change the spectrum (only the phase).

### Rotation
$$f(r, \theta + \theta_0) \Leftrightarrow F(\omega, \psi + \theta_0)$$

Rotating the image by angle $\theta_0$ rotates the spectrum by the same angle.

In [ ]:
from scipy.ndimage import rotate, shift

# Original rectangle
img_orig = np.zeros((256, 256), dtype=np.float64)
img_orig[90:170, 110:150] = 255

# Translated rectangle
img_translated = np.zeros((256, 256), dtype=np.float64)
img_translated[40:120, 150:190] = 255

# Rotated rectangle
img_rotated = rotate(img_orig, 45, reshape=False, order=1)

# Compute spectra
spec_orig = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img_orig))))
spec_translated = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img_translated))))
spec_rotated = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img_rotated))))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img_orig, cmap='gray')
axes[0, 0].set_title('Original', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(img_translated, cmap='gray')
axes[0, 1].set_title('Translated', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(img_rotated, cmap='gray')
axes[0, 2].set_title('Rotated (45 deg)', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(spec_orig, cmap='gray')
axes[1, 0].set_title('Spectrum (Original)', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(spec_translated, cmap='gray')
axes[1, 1].set_title('Spectrum (Translated)\n(Same as original!)', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(spec_rotated, cmap='gray')
axes[1, 2].set_title('Spectrum (Rotated)\n(Also rotated!)', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('DFT Properties: Translation and Rotation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Phase vs Magnitude: Which Carries More Information?

An image can be reconstructed from:
- **Only magnitude** ($|F(u,v)|$, phase set to 0)
- **Only phase** ($\phi(u,v)$, magnitude set to 1)

The **phase** carries most of the **structural/shape** information!
The **magnitude** carries **intensity** information.

In [ ]:
# Create two different images
# Image 1: Rectangle
img1 = np.zeros((256, 256), dtype=np.float64)
img1[70:190, 90:170] = 255

# Image 2: Circle
Y, X = np.ogrid[-128:128, -128:128]
img2 = np.zeros((256, 256), dtype=np.float64)
img2[X**2 + Y**2 <= 50**2] = 255

# Compute DFTs
F1 = np.fft.fft2(img1)
F2 = np.fft.fft2(img2)

# Magnitude and Phase
mag1, phase1 = np.abs(F1), np.angle(F1)
mag2, phase2 = np.abs(F2), np.angle(F2)

# Reconstruct using only magnitude (phase = 0)
recon_mag_only = np.real(np.fft.ifft2(mag1))

# Reconstruct using only phase (magnitude = 1)
recon_phase_only = np.real(np.fft.ifft2(np.exp(1j * phase1)))

# Swap: magnitude of img1 + phase of img2
recon_swap1 = np.real(np.fft.ifft2(mag1 * np.exp(1j * phase2)))

# Swap: magnitude of img2 + phase of img1
recon_swap2 = np.real(np.fft.ifft2(mag2 * np.exp(1j * phase1)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img1, cmap='gray')
axes[0, 0].set_title('Image 1: Rectangle', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(img2, cmap='gray')
axes[0, 1].set_title('Image 2: Circle', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(recon_mag_only, cmap='gray')
axes[0, 2].set_title('Magnitude only (no phase)\nNo structure visible!', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(recon_phase_only, cmap='gray')
axes[1, 0].set_title('Phase only (mag=1)\nRectangle shape visible!', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(recon_swap1, cmap='gray')
axes[1, 1].set_title('Mag(rect) + Phase(circle)\nCircle dominates!', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(recon_swap2, cmap='gray')
axes[1, 2].set_title('Mag(circle) + Phase(rect)\nRectangle dominates!', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Phase vs Magnitude: Phase Carries Shape Information!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Spectrum of Real Images

Let's examine spectra of images with different spatial characteristics.

In [ ]:
# Create images with different spatial features
size = 256

# Horizontal lines
img_h = np.zeros((size, size), dtype=np.float64)
for i in range(0, size, 8):
    img_h[i:i+4, :] = 255

# Vertical lines
img_v = np.zeros((size, size), dtype=np.float64)
for j in range(0, size, 8):
    img_v[:, j:j+4] = 255

# Diagonal pattern
img_d = np.zeros((size, size), dtype=np.float64)
for i in range(size):
    for j in range(size):
        if (i + j) % 16 < 8:
            img_d[i, j] = 255

# Checkerboard
img_c = np.zeros((size, size), dtype=np.float64)
for i in range(size):
    for j in range(size):
        if ((i // 16) + (j // 16)) % 2 == 0:
            img_c[i, j] = 255

images = [img_h, img_v, img_d, img_c]
titles = ['Horizontal Lines', 'Vertical Lines', 'Diagonal Pattern', 'Checkerboard']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (img, title) in enumerate(zip(images, titles)):
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(title, fontsize=11)
    axes[0, i].axis('off')
    
    F = np.fft.fftshift(np.fft.fft2(img))
    axes[1, i].imshow(np.log1p(np.abs(F)), cmap='gray')
    axes[1, i].set_title('Spectrum', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Image', fontsize=12)
axes[1, 0].set_ylabel('Spectrum', fontsize=12)

plt.suptitle('Images and Their Fourier Spectra\n'
             'Note: Horizontal features -> Vertical spectrum, and vice versa',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Symmetry Properties (for real images)

When $f(x,y)$ is **real** (as all images are):

| Property | Spatial Domain | Frequency Domain |
|----------|---------------|------------------|
| Conjugate symmetry | $f(x,y)$ real | $F^*(u,v) = F(-u,-v)$ |
| Real part | $f(x,y)$ real | $R(u,v)$ even, $I(u,v)$ odd |
| Spectrum symmetry | $f(x,y)$ real | $|F(u,v)| = |F(-u,-v)|$ |
| Real & even | $f(x,y)$ real, even | $F(u,v)$ real, even |
| Real & odd | $f(x,y)$ real, odd | $F(u,v)$ imaginary, odd |

In [ ]:
# Demonstrate conjugate symmetry
img = np.random.rand(64, 64)
F = np.fft.fftshift(np.fft.fft2(img))

# F*(u,v) should equal F(-u,-v)
F_conj = np.conj(F)
F_negated = np.flip(F)  # F(-u,-v)

diff = np.max(np.abs(F_conj - F_negated))
print(f"Max difference between F*(u,v) and F(-u,-v): {diff:.2e}")
print("This confirms conjugate symmetry for real images!")

# Spectrum is symmetric
spec = np.abs(F)
spec_flipped = np.flip(spec)
print(f"\nMax difference between |F(u,v)| and |F(-u,-v)|: {np.max(np.abs(spec - spec_flipped)):.2e}")
print("The spectrum is symmetric about the center!")

## Summary

What we learned:
1. **2-D DFT** extends the 1-D DFT to images: $F(u,v) = \sum\sum f(x,y) e^{-j2\pi(ux/M + vy/N)}$
2. **Centering** the DFT (using `fftshift`) moves DC to the center for easier visualization
3. **DC component** $F(0,0)$ is proportional to the average image intensity
4. **Log transformation** is essential for displaying the full spectrum
5. **Translation** does not change the spectrum; **rotation** rotates the spectrum
6. **Phase** carries structural/shape information; **magnitude** carries intensity information
7. **Conjugate symmetry** holds for real images: $|F(u,v)| = |F(-u,-v)|$